<a href="https://colab.research.google.com/github/Cory-Suhr/rush-sales-analysis-final-project/blob/Add-data-files/rush_sales_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GB885 Final Project

## Rush Sales Analysis

Cory Suhr

## Business Problem: You work as a sales analyst for RUSH, a globally renowned sportswear and footwear brand known for its innovative designs and performance-oriented products. The company stores its raw sales data as a collection of three tables:

TABLE_PRODUCTS
TABLE_RETAILER
TABLE_SALES
The data includes the number of units sold, the total sales revenue, the location of the sales, the type of product sold, as well as other relevant information. (For data field definitions and explanations, see the data dictionary.) The data is "raw," meaning it has not been cleaned and probably contains errors that need to be addressed.

The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take initiative to apply your creativity and curiosity to this data.

In addition, she has asked you to answer the following business questions:

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
What state had the highest sales (in dollars) of women's products in 2021? How much was it?
What state had the highest sales (in dollars) of men's products in 2021? How much was it?
What retailer purchased the most units in 2021? In 2020?

## Import Libraries

In [ ]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Loading files from GitHub rather than Google Drive
base_url = (
    'https://raw.githubusercontent.com/'
    'Cory-Suhr/rush-sales-analysis-final-project/'
    'main/'
)

products = pd.read_csv(
    base_url + 'TABLE_PRODUCTS_885.csv',
    sep='|'
)

retailer = pd.read_csv(
    base_url + 'TABLE_RETAILER_885.csv'
)

sales = pd.read_csv(
    base_url + 'TABLE_SALES_885.csv'
)

## Initial Data Inspection

In [ ]:
# Looking at some basics for  products

products.head()

In [ ]:
## Products variable types
products.info()

In [ ]:
# Basics of retailor
retailer.head()

In [ ]:
# Retailer variable types.
retailer.info()

In [ ]:
# Sales basics
sales.head()

In [ ]:
# Sales variable types
sales.info()

## Merge the Data Sets

In [ ]:
# Merging products and sales
sales_products = pd.merge(sales, products, on='PRODUCT_ID', how='left')

In [ ]:
# Checking my work
sales_products.head()

In [ ]:
# Merging sales_products with retailer
combined_df = pd.merge(sales_products, retailer, on='RETAILER_ID', how='left')

In [ ]:
# Check final merger
combined_df.head()

## Looking at the combined dataset for some basics:
Month, Day and Year could be deleted once invoice date is converted to datetime


*   Units sold should be numeric
*   PRICE_PER unit 2 null values
* Retailer, Region, State and City each have 1 null (Drop this row.)
* Sales_Method has 'Oolet' that shoule be 'Outlet'
* PRICE_PER_UNIT has 99999
*



In [ ]:
# Looking at the variable types
combined_df.info()

In [ ]:
# Further looks
combined_df.describe()

In [ ]:
# Looking for null values
combined_df.isnull().sum()

In [ ]:
# Looking at the rows with null values for PRICE_PER_UNIT
combined_df[combined_df['PRICE_PER_UNIT'].isnull()]


In [ ]:
# Looking at null values for RETAILER
combined_df[combined_df['RETAILER'].isnull()]

In [ ]:
# Verifying there this is the only entry with the RETAILER_ID with 999999999
combined_df[combined_df['RETAILER_ID'] == '999999999']

# With the null values in Retailer, region, state and city. And no way to determine the retailer based on the retailer ID going to drop this row from the dataframe

In [ ]:
# Dropping row with index 1534
combind_df = combined_df.drop(index=1534)

In [ ]:
# Looking for Non_traditional categorical data
cat_var = list(combined_df.select_dtypes(include=['object']).columns)
# View unique values for each categorical variable
for column in cat_var:
    print(column)
    print(combined_df[column].unique())


# Look for Duplicate Values

In [ ]:
# Checking for duplicates
combined_df.duplicated().sum()

# Check for Erroneous Data

In [ ]:
# Check for vaule count of erroneous data
# List of categorical variables
cat_var = list(combined_df.select_dtypes(include=['object']).columns)

# Loop through each categorical variable and print the value counts
for column in cat_var:
    print(column)
    print(combined_df[[column]].value_counts())


# Look for Outliers Using the Inner Quantile Ranges (IQR)

In [ ]:
# Write a function to calculate the IQR and print rows with values that fall outside that IQR

def count_iqr_outliers(df, column):
  #define q1
  q1 = df[column].quantile(0.25)
  #define q3
  q3 = df[column].quantile(0.75)
  # define iqr
  iqr = q3- q1
  # define the threshold
  l_threshold = q1 - 1.5 * iqr
  u_threshold = q3 + 1.5 * iqr
  # define outliers
  outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
  # Count the number of True values (outliers)
  return outliers.sum()

In [ ]:
# iterate over the numerical columns of the dataframe
num_var = list(combined_df.select_dtypes(include=['int64', 'float64']).columns)

for column in num_var:
    print(f'{column} : {count_iqr_outliers(combined_df, column)}')

In [ ]:
# Looking into outliers
count_iqr_outliers(combined_df, 'PRICE_PER_UNIT')

## Data Cleaning
-Convert UNITS_SOLD to numeris
- Replace 2 null values in PRICE_PER_UNIT
-Replace the 99999 in PRICE_PER_UNIT
- Replace 'Oolet' with 'Outlet'

In [ ]:
# Converting Units Sold to numeric
combined_df['UNITS_SOLD'] = pd.to_numeric(combined_df['UNITS_SOLD'], errors='coerce')

In [ ]:
# Convert Invoice date to datetime
combined_df['INVOICE_DATE'] = pd.to_datetime(combined_df['INVOICE_DATE'])

In [ ]:
# Checking my work
combined_df.info()

In [ ]:
# Drop Month, Day and Year columns as they provide duplicated information contained in the invoice date
combined_df = combined_df.drop(columns=['MONTH', 'DAY', 'YEAR'])

In [ ]:
# Checking myy work
combined_df.head()

## All null values and the 99999 value in price per unit are with product 20. find the median and mean price for the product and replace the null and erroneous values

In [ ]:
# Finding mean and median for product ID 20
mean_price = combined_df[combined_df['PRODUCT_ID'] == 20]['PRICE_PER_UNIT'].mean()
median_price = combined_df[combined_df['PRODUCT_ID'] == 20]['PRICE_PER_UNIT'].median()

print(mean_price)
print(median_price)

## It seems the 99999 is skewing the mean. I will use the median to repalce the null values and the 99999

In [ ]:
# Replacing null values in price per unit with 44.0

combined_df['PRICE_PER_UNIT'] = combined_df['PRICE_PER_UNIT'].fillna(44.0)

In [ ]:
 # Replacing 99999 with the median of 44.0
combined_df['PRICE_PER_UNIT'] = combined_df['PRICE_PER_UNIT'].replace(99999.000000, 44.0)

## Replacing the 'Ootlet' with 'Oulet'

In [ ]:
# Replace sales method Ootlet with Oulet
combined_df['SALES_METHOD'] = combined_df['SALES_METHOD'].replace('Ootlet', 'Outlet')

In [ ]:
# Checking my work
combined_df.describe()

In [ ]:
# Checking all variable for sales method
combined_df['SALES_METHOD'].unique()

In [ ]:
# Double cheking for null values
combined_df.isnull().sum()

In [ ]:
# Looking at the null values
combined_df[combined_df['RETAILER'].isnull()]

In [ ]:
# Dropping index 1534 again
combined_df = combined_df.drop(index=1534)

In [ ]:
# Checking for null values
combined_df.isnull().sum()

## Exploratory Data Analysis

In [ ]:
# Exploring the cleaned data
combined_df.describe()

In [ ]:
combined_df.select_dtypes(include=['number']).corr()

## VP Business Questions:
The VP of US Sales has tasked you with analyzing sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth. For example, you may want to look for trends or insights in seasonality, retailers, locations, or sales methods. Take initiative to apply your creativity and curiosity to this data.

In addition, she has asked you to answer the following business questions:

What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
What state had the highest sales (in dollars) of women's products in 2021? How much was it?
What state had the highest sales (in dollars) of men's products in 2021? How much was it?
What retailer purchased the most units in 2021? In 2020?

In [ ]:
# Starting with creating a 'REVENUE' column
combined_df['REVENUE'] = combined_df['UNITS_SOLD'] * combined_df['PRICE_PER_UNIT']

In [ ]:
# Looking at the new column
combined_df.head()

## Looking to see which Retailer had the most revenue

In [ ]:
# Revenue by Retailer
combined_df.groupby('RETAILER')['REVENUE'].sum().sort_values(ascending=False)

In [ ]:
# Revue by Region
combined_df.groupby('REGION')['REVENUE'].sum().sort_values(ascending=False)

# Looking at Question 1 What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

In [ ]:
# Looking at 2021
sales_2021 = combined_df[combined_df['INVOICE_DATE'].dt.year == 2021]

In [ ]:
# Grouping by the product name
sales_2021.groupby('PRODUCT_NAME')['REVENUE'].sum().sort_values(ascending=False)

## Question 2: What state had the highest sales (in dollars) of women's products in 2021? How much was it?

In [ ]:
# Filtering by women's products
womens_2021 = sales_2021['PRODUCT_NAME'].str.startswith("Women's")

In [ ]:
# Grouping by state to find the highest sales
state_sales = sales_2021[womens_2021].groupby('STATE')['REVENUE'].sum().sort_values(ascending=False)
state_sales.head(1)

## Question 3: What state had the highest sales (in dollars) of men's products in 2021? How much was it?

In [ ]:
# Looking at men'd products in 2021
mens_2021 = sales_2021['PRODUCT_NAME'].str.startswith("Men's")

In [ ]:
# Grouping men's product revenue by state
mens_state_sales = sales_2021[mens_2021].groupby('STATE')['REVENUE'].sum().sort_values(ascending=False)
mens_state_sales.head(1)

## Question 4: What retailer purchased the most units in 2021? In 2020?

In [ ]:
# Looking at 2020
sales_2020 = combined_df[combined_df['INVOICE_DATE'].dt.year == 2020]

In [ ]:
# Grouping 2021 by retailer
retailer_2021 = sales_2021.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)
retailer_2021.head(1)

In [ ]:
# Grouping 2020 by retailer
retailer_2020 = sales_2020.groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)
retailer_2020.head(1)

In [ ]:
# stopping
combined_df.head()

## Looking at some other insights that could prove worth while for the presentaion.

In [ ]:
# Looking year over year sales data starting with revenue by retailer
retailer_yoy = pd.pivot_table(combined_df,
                              index = 'RETAILER',
                              columns=combined_df['INVOICE_DATE'].dt.year,
                              values='REVENUE',
                              aggfunc='sum')
retailer_yoy


## Was not expecting Null values. Need some further investigating

In [ ]:
# Looking at retailers and year
combined_df.groupby(combined_df['INVOICE_DATE'].dt.year)['RETAILER'].value_counts()

In [ ]:
# Looking at retailers with null values
combined_df[combined_df['RETAILER'] == 'Foot Locker']['INVOICE_DATE'].dt.year.value_counts()

In [ ]:
# Looking at Wal Mart
combined_df[combined_df['RETAILER'] == 'Walmart']['INVOICE_DATE'].dt.year.value_counts()

In [ ]:
# Looking at total revenue year over year
combined_df.groupby(combined_df['INVOICE_DATE'].dt.year)['REVENUE'].sum()

## Footlocker and Walmart are new retailers added for 2021.

## To show retailer growth year over year will need to drop Foot Locker and Walmart for that table

In [ ]:
# Remove retailers that weren't present in both years
retailer_growth = retailer_yoy.dropna().copy()

retailer_growth

In [ ]:
retailer_growth['Growth (%)'] = round((retailer_growth[2021] - retailer_growth[2020]) / retailer_growth[2020] * 100, 2)

# Sort by growth
retailer_growth.sort_values(by='Growth (%)', ascending=False)

## Lookig year over year product sales $

In [ ]:
# Looking at year over year product revenue
product_yoy = pd.pivot_table(combined_df,
                              index = 'PRODUCT_NAME',
                              columns=combined_df['INVOICE_DATE'].dt.year,
                              values='REVENUE',
                              aggfunc='sum')
product_yoy

In [ ]:
# Calculate dollar growth
product_yoy['Growth ($)'] = (product_yoy[2021] - product_yoy[2020])

# Sort by growth
product_yoy.sort_values(by='Growth ($)', ascending=False)

In [ ]:
# Calculate dollar growth %
product_yoy['Growth (%)'] = round((product_yoy[2021] - product_yoy[2020]) / product_yoy[2020] * 100, 2)

# Sort by growth
product_yoy.sort_values(by='Growth (%)', ascending=False)

In [ ]:
# Calculate over all year over year revenue growth
combined_df.groupby(combined_df['INVOICE_DATE'].dt.year)['REVENUE'].sum()

In [ ]:
# Calculate overall dollar growth
combined_df.groupby(combined_df['INVOICE_DATE'].dt.year)['REVENUE'].sum().pct_change() * 100

## Impressive growth was 2020 only a partial year?

In [ ]:
# Looking at the date range
combined_df['INVOICE_DATE'].min(), combined_df['INVOICE_DATE'].max()

In [ ]:
# Looking at revenue by retailer and product for 2021
retailer_product = pd.pivot_table(
    sales_2021,
    index = 'RETAILER',
    columns = 'PRODUCT_NAME',
    values ='REVENUE',
    aggfunc ='sum'
)
retailer_product

In [ ]:
# Convert each retailer's product sales to a percentage of its total sales
retailer_product_pct = (retailer_product.div(retailer_product.sum(axis =1 ), axis=0) *100).round(1)

retailer_product_pct

## Visualizations.
1. Revenue by Product (2021)
2. Product Growth (2020 - to 2021)
3. Retailer Revenue Growth
4. Retailer Reveenue (2020 vs 2021)
5. Retailer Product Mix
6. State Revenue

In [ ]:
# create product_sales
product_sales = (sales_2021.groupby('PRODUCT_NAME')['REVENUE'].sum().sort_values(ascending=False))
product_sales

In [ ]:
# Bring in formatter to format axis
from matplotlib.ticker import FuncFormatter



## Need to create folders in Colab and Github to store chart images for the presentation

In [ ]:
# Create an images folder in Colab
import os

os.makedirs('images', exist_ok=True)

In [ ]:
# Chart 1 revenue by product 2021 Horizontal bar chart
product_sales.sort_values().plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('2021 Revenue by Product Category')
plt.xlabel('Revenue (Millions $)')
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1_000_000:.0f}M'))
plt.ylabel('')
plt.tight_layout()
plt.savefig('images/chart1_product_revenue_2021.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 2 product growth 2020 to 2021 horizontal bar chart
product_yoy['Growth ($)'].sort_values().plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('Year over Year Revenue Growth by Product')
plt.xlabel('Revenue Growth (Millions $)')
plt.ylabel('')
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1_000_000: .0f}M'))
plt.tight_layout()
plt.savefig('images/chart2_product_growth.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 3 Retailer Revenue Growth
retailer_growth['Growth ($)'] = (retailer_growth[2021] - retailer_growth[2020])
retailer_growth['Growth ($)'].sort_values().plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('Retailer Revenue Growth (2020-2021)')
plt.xlabel('Revenue Growth (Millions $)')
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1_000_000: .0f}M'))
plt.axvline(x=0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig('images/chart3_retailer_revenue_growth.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 4 Retailer Revenue (2020 vs 2021)
retailer_yoy = retailer_yoy.sort_values(by=2021, ascending=False)
retailer_yoy.plot(
    kind='barh',
    figsize=(8,5)
)

plt.title('Retailer Revenue by Year')
plt.xlabel('Revenue ( Millions $)')
plt.ylabel('')
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1_000_000: .0f}M'))
plt.legend(title= 'Year')
plt.tight_layout()
plt.savefig('images/chart4_retailer_revenue_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 5 retailer Product Mix
ax = retailer_product_pct.plot(
    kind='barh',
    stacked=True,
    figsize=(10,6)
)

plt.title('Retailer Product Mix (2021)')
plt.xlabel('Percent of Retailer Revenue')
plt.ylabel('')
plt.xlim(0, 100)
plt.legend(title='Product Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('images/chart5_retailer_product_mix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 6 State Revenue
state_sales.sort_values().plot(
    kind='barh',
    figsize=(8,6)
)

plt.title("Women's Product Revenue by State (2021)")
plt.tight_layout()

## Showing all 50 states is too crowded. Simplifying down to top 10.

In [ ]:
# Top 10 states for women's product revenue in 2021
top10_state_sales = state_sales.nlargest(10)

top10_state_sales.sort_values().plot(
    kind='barh',
    figsize=(8,5)
)
plt.title("Top 10 States for Women's Product Revenue (2021)")
plt.xlabel('Revenue (Millions $)')
plt.ylabel('')

plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1_000_000: .1f}M'))
plt.tight_layout()
plt.savefig('images/chart6_top10_womens_revenue_2021.png', dpi=300, bbox_inches='tight')
plt.show()




## Downloading all created images in a zip file

In [ ]:
# Downloading images
import shutil

shutil.make_archive('Rush_Sales_Charts', 'zip', 'images')

In [ ]:
# Saving images to computer
from google.colab import files

files.download('Rush_Sales_Charts.zip')